# 5e Compare Experiments

Use a validation split to compare regularization settings, then test one selected model. Record results locally and optionally log selected metrics to ClearML.

**Budget:** 15–25 minutes; a CPU and a few thousand images are enough. No Klique endpoint or token is needed for the default path.

**Before you start:** complete the [JupyterLab introduction](05_Running_Jupyter_Lab.ipynb), prepare Python 3.10+ and the packages in the [support guide](README.md), and select that kernel. Run every cell from a fresh kernel in order. The default is offline: cached real datasets are used when present; otherwise the helper explicitly identifies an 8×8 scikit-learn digits fallback. That fallback is **not MNIST, EMNIST, or Fashion-MNIST**, and its metrics are not comparable to theirs. Downloads are opt-in; read [dataset sources and terms](DATASETS.md) first.

Save your work in the approved persistent workspace. No live Klique launch or restore is established by these local exercises.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
LAB = HERE if (HERE / "image_lab.py").is_file() else HERE / "05_Jupyter"
if not (LAB / "image_lab.py").is_file():
    raise RuntimeError("Start this kernel in the repository root or 05_Jupyter folder.")
if str(LAB) not in sys.path:
    sys.path.insert(0, str(LAB))
from image_lab import load_image_data, train_baseline
SEED = 42
OUTPUT = LAB / "lab_outputs"
OUTPUT.mkdir(exist_ok=True)


In [ ]:
import json
from dataclasses import replace
from sklearn.model_selection import train_test_split
import pandas as pd

data = load_image_data("mnist", allow_download=False,
                       cache_dir=OUTPUT / "data", limit_train=2400, limit_test=500, seed=SEED)
train_idx, valid_idx = train_test_split(np.arange(len(data.train_labels)),
    test_size=0.25, stratify=data.train_labels, random_state=SEED)
validation_data = replace(data,
    train_images=data.train_images[train_idx], train_labels=data.train_labels[train_idx],
    test_images=data.train_images[valid_idx], test_labels=data.train_labels[valid_idx])
print("Actual dataset:", data.name)
print("Fit / validation / untouched test:", len(train_idx), len(valid_idx), len(data.test_labels))


## Select with validation, not test

Use the same split and random seed for every candidate. Compare validation accuracy and fit time, keeping the search bounded. The official test labels are not consulted until selection finishes. A small validation difference may be sampling noise; this is not a statistical significance claim.


In [ ]:
rows = []
for regularization in [0.1, 1.0, 10.0]:
    candidate, candidate_metrics = train_baseline(validation_data, C=regularization,
                                                 max_iter=100, seed=SEED)
    score = float(candidate.score(validation_data.test_images.reshape(len(valid_idx), -1),
                                  validation_data.test_labels))
    rows.append({"C": regularization, "validation_accuracy": score,
                 "dataset": data.name, "seed": SEED})
comparison = pd.DataFrame(rows).sort_values(["validation_accuracy", "C"], ascending=[False, True])
display(comparison)
best_c = float(comparison.iloc[0]["C"])
final_model, final_metrics = train_baseline(data, C=best_c, max_iter=100, seed=SEED)
print("Selected C:", best_c, "Final held-out test accuracy:", final_metrics["accuracy"])
comparison.to_csv(OUTPUT / "experiment_comparison.csv", index=False)
(OUTPUT / "selected_experiment.json").write_text(json.dumps({
    "dataset": data.name, "seed": SEED, "selected_C": best_c,
    "test_accuracy": float(final_metrics["accuracy"]),
    "selection": "stratified 25% validation from training only"}, indent=2))


## Optional ClearML logging

Local CSV/JSON files are the default. Installing the SDK is optional. Set `LOG_TO_CLEARML = True` only after your administrator approves the server, project, credentials, and the metrics below leaving this workspace. Use approved secret injection or the SDK's documented configuration, never a token pasted into a cell. This logs only the named public-data parameters and scalar results, disables automatic framework/argument/resource/console logging, and does not upload images or model files. The SDK/service may still record account and execution metadata; inspect your deployment's policy. It is not proof of Klique or ClearML tenancy connectivity.


In [ ]:
LOG_TO_CLEARML = False
if LOG_TO_CLEARML:
    from clearml import Task
    task = Task.init(project_name="Jupyter teaching labs", task_name="05e validation comparison",
                     reuse_last_task_id=False, auto_connect_frameworks=False,
                     auto_connect_arg_parser=False, auto_resource_monitoring=False,
                     auto_connect_streams=False,
                     output_uri=False)
    try:
        task.connect({"dataset": data.name, "seed": SEED, "selected_C": best_c}, name="approved_parameters")
        logger = task.get_logger()
        for index, row in enumerate(rows):
            logger.report_scalar("validation_accuracy", f"C={row['C']}",
                                 value=row["validation_accuracy"], iteration=index)
        logger.report_scalar("held_out_test_accuracy", "selected_model",
                             value=float(final_metrics["accuracy"]), iteration=0)
    finally:
        task.close()
else:
    print("ClearML is off. Results remain in local lab_outputs.")


## Checkpoint, exercise, and cleanup

**Checkpoint:** Three candidates share one validation split; a single selected configuration is evaluated on the held-out test set and results are saved locally.

**Try it:** Change one training-only candidate setting and rerun the selection. Explain which metric chose the model and which metric was reserved for final reporting.

**Troubleshooting:** if imports fail, select the kernel where the section requirements were installed. If a data host is blocked, keep downloads off and use the named offline fixture, or ask your administrator to stage the official data in the cache. Do not disable TLS verification. A convergence warning means the solver hit its iteration budget; record it rather than silently claiming an optimum.

Save the notebook and useful artifacts under `lab_outputs/`; it is ignored by Git but may still enter workspace snapshots. Clear outputs before sharing if you used private data. Shut down unused kernels, then stop the managed session and verify its stopped state as in the introduction. Closing a browser tab is not cleanup.

**Next:** [5f: Draw a Digit](05f_Draw_a_Digit.ipynb) · **Index:** [Jupyter labs](README.md) · [Orientation](../00_Orientation/00_Orientation.ipynb)
